In [1]:
# ==============================================================================
# NOTEBOOK 05: PRESCRIPTIVE TREASURY WORKING CAPITAL OPTIMIZATION (MILP)
# ==============================================================================
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from pyspark.sql import functions as F

# 1. Load Latest Real Macro Rates from Silver Layer
pdf_macro = spark.read.table("silver_macro_features").toPandas()
latest_macro = pdf_macro.sort_values("observation_date").iloc[-1]

fed_funds_rate = float(latest_macro["FEDFUNDS"]) / 100.0
t_bill_rate = float(latest_macro["DGS2"]) / 100.0

# 2. Define Treasury Liquidity Instruments ($500M Total Daily Operational Float)
total_float = 500.0  # $500 Million

instruments = [
    {
        "instrument_id": "INST-01",
        "instrument_name": "Fed Overnight Reverse Repo (ON RRP)",
        "tier": "Tier-1 Liquid Sweep",
        "annual_yield": round(fed_funds_rate * 0.96, 5),  # ~96% of Fed Funds
        "setup_cost_m": 0.05,
        "min_tranche_m": 25.0,
        "max_cap_m": 175.0,  # 35% counterparty concentration ceiling
        "legacy_alloc_m": 110.0
    },
    {
        "instrument_id": "INST-02",
        "instrument_name": "US 3-Month Sovereign T-Bills",
        "tier": "Short-Term Sovereign Paper",
        "annual_yield": round(t_bill_rate * 1.02, 5),
        "setup_cost_m": 0.08,
        "min_tranche_m": 50.0,
        "max_cap_m": 175.0,  # 35% concentration ceiling
        "legacy_alloc_m": 70.0
    },
    {
        "instrument_id": "INST-03",
        "instrument_name": "Partner Bank Sweeps (Tier-1 Money Market)",
        "tier": "Intraday Callable Liquidity",
        "annual_yield": round(fed_funds_rate * 0.82, 5),
        "setup_cost_m": 0.03,
        "min_tranche_m": 20.0,
        "max_cap_m": 175.0,
        "legacy_alloc_m": 0.0
    },
    {
        "instrument_id": "INST-04",
        "instrument_name": "Commercial Clearing Cash Buffer (Zero-Yield)",
        "tier": "Statutory Intraday Settlement",
        "annual_yield": 0.0015,  # 15 bps nominal earnings credit
        "setup_cost_m": 0.00,
        "min_tranche_m": 85.0,   # Basel III LCR 99.9% settlement finality floor ($85M)
        "max_cap_m": 500.0,
        "legacy_alloc_m": 320.0  # Legacy bloated idle cash!
    }
]

df_inst = pd.DataFrame(instruments)
n = len(df_inst)

# 3. Formulate Mixed-Integer Linear Program (MILP)
# Decision vector x = [alloc_1..alloc_4, binary_active_1..binary_active_4]
# Maximize: sum(alloc_i * yield_i - setup_cost_i * binary_i)
# SciPy milp minimizes c^T * x, so negate objective coefficients:
c = np.concatenate([
    -df_inst["annual_yield"].values,
    df_inst["setup_cost_m"].values
])

integrality = np.array([0]*n + [1]*n)  # 0 = continuous ($M), 1 = binary activation

# Bounds: 0 <= alloc_i <= max_cap_i; 0 <= binary_i <= 1
lb = np.concatenate([np.zeros(n), np.zeros(n)])
ub = np.concatenate([df_inst["max_cap_m"].values, np.ones(n)])
bounds = Bounds(lb, ub)

# Linear Constraints:
# Constraint 1: Sum of all allocations == $500.0M
A_budget = np.zeros((1, 2*n))
A_budget[0, :n] = 1.0

# Constraint 2..5: Minimum tranche activation: alloc_i - min_tranche_i * binary_i >= 0
A_min = np.zeros((n, 2*n))
for i in range(n):
    A_min[i, i] = 1.0
    A_min[i, n + i] = -df_inst.loc[i, "min_tranche_m"]

# Constraint 6: Force binary_4 = 1 (Statutory clearing buffer must always remain active)
A_clear = np.zeros((1, 2*n))
A_clear[0, n + 3] = 1.0

A_all = np.vstack([A_budget, A_min, A_clear])
b_l = np.concatenate([[total_float], np.zeros(n), [1.0]])
b_u = np.concatenate([[total_float], np.full(n, np.inf), [1.0]])

constraints = LinearConstraint(A_all, b_l, b_u)

# 4. Solve MILP using HiGHS Engine
res = milp(c=c, integrality=integrality, bounds=bounds, constraints=constraints)

if res.success:
    df_inst["optimal_alloc_m"] = np.round(res.x[:n], 2)
    df_inst["is_active"] = np.round(res.x[n:]).astype(int)
    df_inst["legacy_annual_yield_m"] = np.round(df_inst["legacy_alloc_m"] * df_inst["annual_yield"], 3)
    df_inst["optimal_annual_yield_m"] = np.round(
        df_inst["optimal_alloc_m"] * df_inst["annual_yield"] - (df_inst["setup_cost_m"] * df_inst["is_active"]), 3
    )
    df_inst["delta_alloc_m"] = np.round(df_inst["optimal_alloc_m"] - df_inst["legacy_alloc_m"], 2)
    df_inst["delta_yield_m"] = np.round(df_inst["optimal_annual_yield_m"] - df_inst["legacy_annual_yield_m"], 3)

    legacy_total_yield = df_inst["legacy_annual_yield_m"].sum()
    optimal_total_yield = df_inst["optimal_annual_yield_m"].sum()
    net_gain = optimal_total_yield - legacy_total_yield

    print("=== PRESCRIPTIVE TREASURY MILP OPTIMIZATION RESULTS ===")
    print(f"Solver Status:             Optimal Solution Found (HiGHS)")
    print(f"Legacy Idle Clearing Cash: ${df_inst.loc[3, 'legacy_alloc_m']:.2f}M -> Optimized: ${df_inst.loc[3, 'optimal_alloc_m']:.2f}M")
    print(f"Legacy Annual Portfolio Yield:  ${legacy_total_yield:.2f}M")
    print(f"Optimal Annual Portfolio Yield: ${optimal_total_yield:.2f}M")
    print(f"Net Annual Yield Uplift:       +${net_gain:.2f}M / year")

    sdf_treasury = spark.createDataFrame(df_inst) \
        .withColumn("optimized_at", F.current_timestamp())
    sdf_treasury.write.format("delta").mode("overwrite").saveAsTable("gold_fact_treasury_optimization")
    print("Saved gold_fact_treasury_optimization to OneLake.")
    display(sdf_treasury)
else:
    print("MILP Optimization failed:", res.message)


StatementMeta(, b8ab88aa-d3ee-4c8d-9683-f54a29285286, 3, Finished, Available, Finished, False)

=== PRESCRIPTIVE TREASURY MILP OPTIMIZATION RESULTS ===
Solver Status:             Optimal Solution Found (HiGHS)
Legacy Idle Clearing Cash: $320.00M -> Optimized: $85.00M
Legacy Annual Portfolio Yield:  $7.83M
Optimal Annual Portfolio Yield: $16.85M
Net Annual Yield Uplift:       +$9.02M / year
Saved gold_fact_treasury_optimization to OneLake.


SynapseWidget(Synapse.DataFrame, e3a16355-9539-4889-9c0e-3ca330ce5945)